# Build Dataframe with CDS

In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np
from fuzzywuzzy import process

## Load the CDS data

In [ ]:
cds_filename = 'CDS//cds.csv'
dfcds = pd.read_csv(cds_filename)

In [ ]:
dfcds.head()

In [ ]:
tickers = dfcds['Ticker'].unique().tolist()
companies = dfcds['Company'].unique().tolist()

In [ ]:
def get_tickers(company_name, comp_dict, threshold=90):
    match, similarity = process.extractOne(company_name, comp_dict.keys())
    if similarity >= threshold:
        return comp_dict[match]
    else:
        raise Exception("Not found")

In [ ]:
def buildCompanyDataframe(tickers, companies, name_ticker_map):
    ticker_list = list()
    name_list = list()
    country_list = list()
    sector_list = list()
    for itkr, icomp in zip(tickers, companies):
        yftkr = yf.Ticker(itkr)
        tkr_info = None
        try:
            tkr_info = yftkr.info
        except:
            try:
                tkr_lst = get_tickers(icomp, name_ticker_map, threshold=70)
                for trytkrs in tkr_lst:
                    try:
                        yftkr = yf.Ticker(trytkrs)
                        tkr_info = yftkr.info
                    except:
                        continue        
            except:
                continue
                
        if tkr_info != None:
            try:
                country = tkr_info['country']
                sector = tkr_info['sector']
                if country != "" and sector != "":
                    ticker_list.append(itkr)
                    country_list.append(country)
                    sector_list.append(sector)
                    name_list.append(icomp)
            except:
                continue

    return ticker_list, country_list, sector_list, name_list

In [ ]:
from pytickersymbols import PyTickerSymbols

stock_data = PyTickerSymbols()

In [ ]:
uk_stocks = stock_data.get_stocks_by_index('FTSE 100')
german_stocks = stock_data.get_stocks_by_index('DAX')
french_stocks = stock_data.get_stocks_by_index('CAC 40')
dj_stocks = stock_data.get_stocks_by_index('DOW JONES')
nasdaq = stock_data.get_stocks_by_index('NASDAQ 100')
sp500 = stock_data.get_stocks_by_index('S&P 500')

In [ ]:
stk_list = list(uk_stocks) + list(german_stocks) + list(french_stocks) + \
    list(dj_stocks) + list(nasdaq) + list(sp500)

In [ ]:
stk_list

In [ ]:
name_ticker_map = {
    elem['name']: [tkr['yahoo'] for tkr in elem['symbols']]
    for elem in stk_list
    if 'symbols' in elem
}

In [ ]:
t_list, c_list, s_list, n_list = buildCompanyDataframe(tickers, 
                                                       companies, name_ticker_map)

In [ ]:
comp_dict = {'Ticker': t_list, 'Company': n_list, 
             'Sector': s_list, 'Country': c_list}

In [ ]:
comp_df = pd.DataFrame(comp_dict)

In [ ]:
comp_df.tail(20)

In [ ]:
spcorp_file = '20220601 Standard & Poor\'s Ratings Services Corporate.csv'
spfin_file = '20220601 Standard & Poor\'s Ratings Services Financial.csv'

In [ ]:
spcorp_df = pd.read_csv(spcorp_file)

In [ ]:
spfin_df = pd.read_csv(spfin_file)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
spcorp_names = spcorp_df['obligor_name'].unique().tolist()

In [ ]:
def getRating(match, df):
    dfsub = df[df['obligor_name'] == match]
    dfsubsub = dfsub[dfsub['rating_sub_type'] == 'Local Currency LT']
    if len(dfsubsub) > 0:
        return dfsubsub['rating'].iloc[0]
    elif len(dfsub) > 0:
        return dfsub['rating'].iloc[0]
    else:
        return ''

In [ ]:
def buildRatingsDict(companies, spcorp_df, spfin_df):
    rating_dict = dict()
    spcorp_names = spcorp_df['obligor_name'].unique().tolist()
    spfin_names = spfin_df['obligor_name'].unique().tolist()
    for icomp in companies:
        match, similarity = process.extractOne(icomp, spcorp_names)
        if similarity >= 90:
            rating_dict[icomp] = getRating(match, spcorp_df)
        else:
            match, similarity = process.extractOne(icomp, spfin_names)
            if similarity >= 90:
                rating_dict[icomp] = getRating(match, spfin_df)
            else:
                continue
    return rating_dict

In [ ]:
rating_dict = buildRatingsDict(companies, spcorp_df, spfin_df)

In [ ]:
def remove_after_slash(input_string):
    parts = input_string.split('/', 1)
    return parts[0]

def remove_substring(original_string, substring):
    # Replace the substring with an empty string
    return original_string.replace(substring, '')

In [ ]:
revised_rating_dict = dict()
for key, value in rating_dict.items():
    revised_rating_dict[key] = remove_substring(remove_after_slash(value), 'mx')

In [ ]:
revised_rating_dict

In [ ]:
comp_df['Rating'] = comp_df['Company'].map(revised_rating_dict)

In [ ]:
comp_df

In [ ]:
redcomp_df = comp_df.dropna()

In [ ]:
redcomp_df = redcomp_df[redcomp_df['Rating'] != '']

In [ ]:
redcomp_df

In [ ]:
combined_df = dfcds.merge(redcomp_df, on='Company', how='left')

In [ ]:
combined_df

In [ ]:
redcomb_final = combined_df.dropna()

In [ ]:
redcomb_final

In [ ]:
cds_data = redcomb_final.drop('Ticker_y', axis=1)

In [ ]:
cds_data.rename(columns={'Ticker_x': 'Ticker'}, inplace=True)

In [ ]:
cds_data

In [ ]:
output_filename = 'CDSRatSecGeo.csv'

In [ ]:
cds_data.to_csv(output_filename)

In [ ]:
len(cds_data['Company'].unique())